# 1. CẤU HÌNH FILE

In [1]:
import pandas as pd
from pathlib import Path

INPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/10_occupation_taxonomy_clean.xlsx"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/11_occupation_taxonomy_tree_final.xlsx"

# 2. HÀM ĐỌC FILE

In [2]:
def read_table(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng file chưa hỗ trợ: {path.suffix}")

# 3. HÀM TÌM CỘT


In [3]:
def find_existing_column(df: pd.DataFrame, candidates: list[str], required=True):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise KeyError(
            f"Không tìm thấy cột phù hợp. Candidates={candidates}. "
            f"Các cột hiện có: {list(df.columns)}"
        )
    return None

# 4. TẠO NODE

In [4]:
def make_node(node_id, parent_id, level, node_name, node_type,
              taxonomy_group=None, taxonomy_subgroup=None,
              occupation_id=None, occupation_name=None,
              group_value=None, skill_count=None):
    return {
        "node_id": node_id,
        "parent_id": parent_id,
        "level": level,
        "node_name": node_name,
        "node_type": node_type,   # root / group / subgroup / occupation
        "taxonomy_group": taxonomy_group,
        "taxonomy_subgroup": taxonomy_subgroup,
        "occupation_id": occupation_id,
        "occupation_name": occupation_name,
        "group": group_value,
        "skill_count": skill_count
    }


# 5. CHẠY CHÍNH

Bước 4 lần lượt là:

1. đọc file clean

2. lấy các cột cần dùng

3. chuẩn hóa tên cột

4. sắp xếp dữ liệu

5. tạo node gốc

6. tạo node group lớn

7. tạo node subgroup

8. tạo node occupation

9. ghép tất cả thành cây

10. xuất ra file excel cuối

In [5]:
def main():
    # Đọc file clean
    df = read_table(INPUT_FILE)
    print("Đã đọc file:", df.shape)
    print("Các cột hiện có:", df.columns.tolist())

    # Tìm các cột cần thiết
    occ_id_col = find_existing_column(df, ["occupation_id"], required=False)
    occ_name_col = find_existing_column(df, ["occupation_name", "preferredLabel", "occupationLabel"])
    group_col = find_existing_column(df, ["group"], required=False)
    skill_count_col = find_existing_column(df, ["skill_count", "num_skills"], required=False)
    taxonomy_group_col = find_existing_column(df, ["taxonomy_group"])
    taxonomy_subgroup_col = find_existing_column(df, ["taxonomy_subgroup"])

    # Đổi tên cột cho dễ xử lý
    rename_map = {
        occ_name_col: "occupation_name",
        taxonomy_group_col: "taxonomy_group",
        taxonomy_subgroup_col: "taxonomy_subgroup",
    }

    if occ_id_col:
        rename_map[occ_id_col] = "occupation_id"
    if group_col:
        rename_map[group_col] = "group"
    if skill_count_col:
        rename_map[skill_count_col] = "skill_count"

    df = df.rename(columns=rename_map).copy()

    # Nếu thiếu cột thì thêm cho đủ
    if "occupation_id" not in df.columns:
        df["occupation_id"] = None
    if "group" not in df.columns:
        df["group"] = None
    if "skill_count" not in df.columns:
        df["skill_count"] = None

    # Sắp xếp cho dễ nhìn
    df = df.sort_values(
        by=["taxonomy_group", "taxonomy_subgroup", "occupation_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    nodes = []

    # ROOT
    root_id = "root_1"
    nodes.append(
        make_node(
            node_id=root_id,
            parent_id=None,
            level=0,
            node_name="IT Occupation Taxonomy",
            node_type="root"
        )
    )

    group_counter = 1
    subgroup_counter = 1
    occupation_counter = 1

    group_id_map = {}
    subgroup_id_map = {}

    # Tạo node group
    unique_groups = df["taxonomy_group"].dropna().drop_duplicates().tolist()
    for tg in unique_groups:
        group_id = f"group_{group_counter}"
        group_id_map[tg] = group_id

        nodes.append(
            make_node(
                node_id=group_id,
                parent_id=root_id,
                level=1,
                node_name=tg,
                node_type="group",
                taxonomy_group=tg
            )
        )
        group_counter += 1

    # Tạo node subgroup
    unique_group_subgroups = (
        df[["taxonomy_group", "taxonomy_subgroup"]]
        .dropna()
        .drop_duplicates()
        .sort_values(by=["taxonomy_group", "taxonomy_subgroup"])
        .values.tolist()
    )

    for tg, tsg in unique_group_subgroups:
        subgroup_id = f"subgroup_{subgroup_counter}"
        subgroup_id_map[(tg, tsg)] = subgroup_id

        nodes.append(
            make_node(
                node_id=subgroup_id,
                parent_id=group_id_map[tg],
                level=2,
                node_name=tsg,
                node_type="subgroup",
                taxonomy_group=tg,
                taxonomy_subgroup=tsg
            )
        )
        subgroup_counter += 1

    # Tạo node occupation
    for _, row in df.iterrows():
        occ_node_id = f"occupation_{occupation_counter}"
        occupation_counter += 1

        parent_id = subgroup_id_map[(row["taxonomy_group"], row["taxonomy_subgroup"])]

        nodes.append(
            make_node(
                node_id=occ_node_id,
                parent_id=parent_id,
                level=3,
                node_name=row["occupation_name"],
                node_type="occupation",
                taxonomy_group=row["taxonomy_group"],
                taxonomy_subgroup=row["taxonomy_subgroup"],
                occupation_id=row["occupation_id"],
                occupation_name=row["occupation_name"],
                group_value=row["group"],
                skill_count=row["skill_count"]
            )
        )

    # Tạo dataframe cây taxonomy
    df_tree = pd.DataFrame(nodes)

    # Xuất file excel
    output_path = Path(OUTPUT_FILE)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_tree.to_excel(writer, sheet_name="taxonomy_tree", index=False)
        df.to_excel(writer, sheet_name="occupation_clean", index=False)

    print(f"Đã tạo file: {OUTPUT_FILE}")
    print("Tổng số node:", len(df_tree))
    print(df_tree.head(20))


if __name__ == "__main__":
    main()

Đã đọc file: (54, 23)
Các cột hiện có: ['occupation_id', 'occupation_name', 'group', 'skill_count', 'taxonomy_group', 'taxonomy_subgroup', 'conceptType', 'iscoGroup', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode', 'title_clean', 'is_excluded', 'is_core', 'is_extended']
Đã tạo file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/11_occupation_taxonomy_tree_final.xlsx
Tổng số node: 81
        node_id parent_id  level                      node_name node_type  \
0        root_1      None      0         IT Occupation Taxonomy      root   
1       group_1    root_1      1                      Data & AI     group   
2       group_2    root_1      1                         Design     group   
3       group_3    root_1      1         Infrastructure & Cloud     group   
4       group_4    root_1      1         